In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
import os 

client = OpenAI(
    api_key = os.getenv("GROQ_API_KEY"),
    base_url = "https://api.groq.com/openai/v1"
)


In [3]:
def llm(prompt):
    response = client.responses.create(
        model = "llama-3.1-8b-instant",
        input = prompt
    )
    return response.output_text

In [4]:
llm("Hey, What's up?")

'Not much. Just here to help and chat. How can I assist you today?'

In [5]:
question = "I just discovered this course. Can i join now?" 
answer = llm(question)
print(answer)

I'm happy to help, but I'm a large language model, I don't have a specific course to join you in. It seems that our conversation has just started. Could you please tell me more about the course you're interested in? I'll do my best to provide you with information on how to join or get started.


In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [7]:
prompt = f"""
Your task is to answer the questions from the course participants based on the provided context.

Use the context to find relevant information and provide accurate answers. If the answer is not found in the context, respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [8]:
answer = llm(prompt)
print(answer)

Based on the provided context, I'll answer your question:

I just discovered this course. Can i join now?

You can still join the course, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [9]:
import requests
import json

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()
print(json.dumps(courses_raw, indent=2))

[
  {
    "course": "data-engineering-zoomcamp",
    "course_name": "Data Engineering Zoomcamp",
    "path": "/json/data-engineering-zoomcamp.json",
    "questions_count": 404
  },
  {
    "course": "stock-markets-analytics-zoomcamp",
    "course_name": "Stock Markets Analytics Zoomcamp",
    "path": "/json/stock-markets-analytics-zoomcamp.json",
    "questions_count": 93
  },
  {
    "course": "ai-dev-tools-zoomcamp",
    "course_name": "AI Dev Tools Zoomcamp",
    "path": "/json/ai-dev-tools-zoomcamp.json",
    "questions_count": 41
  },
  {
    "course": "llm-zoomcamp",
    "course_name": "LLM Zoomcamp",
    "path": "/json/llm-zoomcamp.json",
    "questions_count": 84
  },
  {
    "course": "mlops-zoomcamp",
    "course_name": "MLOps Zoomcamp",
    "path": "/json/mlops-zoomcamp.json",
    "questions_count": 255
  },
  {
    "course": "machine-learning-zoomcamp",
    "course_name": "ML Zoomcamp",
    "path": "/json/machine-learning-zoomcamp.json",
    "questions_count": 472
  }
]


In [10]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1349

In [11]:
documents[100]

{'id': 'f291e8d311',
 'course': 'data-engineering-zoomcamp',
 'section': 'Module 1: Postgres, pgAdmin & Python ingestion',
 'question': 'Postgres: bind: address already in use',
 'answer': 'When attempting to start the Docker Postgres container, you may encounter the error message:\n\n```\nError - postgres port is already in use.\n```\n\n**Option 1: Identify and Stop the Service**\n\n1. Determine which service is using the port by running:\n   \n   ```bash\n   sudo lsof -i :5432\n   ```\n   \n2. Stop the service that is using the port:\n   \n   ```bash\n   sudo service postgresql stop\n   ```\n\n**Option 2: Map to a Different Port**\n\nFor a more long-term solution, consider mapping to a different port:\n\n- Map local port 5433 to container port 5432 in your Docker configuration (`Dockerfile` or `docker-compose.yml`).\n- If using a VM, ensure that port 5433 is forwarded in the host machine configuration.\n\nThis approach prevents conflicts and allows the Docker Postgres container to ru

In [12]:
from minsearch import Index

index = Index(
    text_fields = ["question", "answer", "section"],
    keyword_fields = ["course"]
)

index.fit(documents)

In [13]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict = boost_dict,
        filter_dict = filter_dict,
        num_results = 5
    )

In [14]:
search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [15]:
INSTRUCTIONS = """
Your task is to answer the questions from the course participants based on the provided context.

Use the context to find relevant information and provide accurate answers. If the answer is not found in the context, respond with "I don't know."
"""

In [16]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [17]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [18]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question = question,
        context = context
    )

    return prompt.strip()

In [19]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
I just discovered this course. Can i join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your projec

In [20]:
response = client.responses.create(
    model = "llama-3.1-8b-instant",
    input = prompt
)

In [21]:
response.output_text

'Yes, you can join the course now. The relevant text is: "You can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/)."'

In [22]:
response.usage

ResponseUsage(input_tokens=510, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=62, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=572)

In [24]:
def llm(instructions, user_prompt, model = "llama-3.1-8b-instant"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = client.responses.create(
        model = model,
        input = message_history
    )

    return response.output_text

In [25]:
def rag(query, model = "llama-3.1-8b-instant"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model = model)

    return answer


In [26]:
answer = rag("How do I get a certificate?")
print(answer)

To get a certificate, you must:

1. **Finish the course with a "live" cohort**: You can only get a certificate if you participate in a live cohort, not in a self-paced mode.

2. **Pass the Capstone project**: Even if you miss the first homework, you can still get a certificate by passing the Capstone project.
